# TradeFind — AI Import/Export Directory
### AI/ML Internship Project | Production Ready

**3 simple steps:**

| Step | Cell | What it does |
|---|---|---|
| 1 | Cell 1 | Install libraries + cloudflared |
| 2 | Cell 2 | Build app + launch Flask |
| 3 | Cell 3 | Get your live public URL |

**Requirements:**
- Save your Groq API key in Colab Secrets with name: `API`
- Get free key at: https://console.groq.com

In [ ]:
# ═══════════════════════════════════════════════════
# CELL 1 — Install libraries  (run once, ~2 min)
# ═══════════════════════════════════════════════════
!pip install flask sentence-transformers scikit-learn openpyxl groq --quiet

# Install cloudflared for public URL (no account needed)
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared

print('All libraries and tools installed!')

In [ ]:
# ═══════════════════════════════════════════════════
# CELL 2 — Build the full app
# Reads Groq key from Colab Secrets (key name: API)
# Prints 1/6 to 6/6 as it builds each part
# ═══════════════════════════════════════════════════
import os, sqlite3, subprocess, threading, time, sys, pathlib
from google.colab import userdata

# Load Groq API key from Colab Secrets
os.environ['GROQ_API_KEY'] = userdata.get('API')
print('Groq key loaded!')

# 1 — Folders
for d in ['/content/tradefind/templates','/content/tradefind/static/css',
          '/content/tradefind/static/js','/content/tradefind/data','/content/tradefind/ai']:
    os.makedirs(d, exist_ok=True)
open('/content/tradefind/ai/__init__.py','w').close()
print('1/6 Folders ready')

# 2 — Database
conn=sqlite3.connect('/content/tradefind/data/trade.db'); cur=conn.cursor()
cur.execute('DROP TABLE IF EXISTS companies')
cur.execute('''CREATE TABLE companies(id INTEGER PRIMARY KEY AUTOINCREMENT,
  company_name TEXT,contact_person TEXT,phone1 TEXT,phone2 TEXT,mobile TEXT,
  email1 TEXT,email2 TEXT,website TEXT,address TEXT,products TEXT,
  services TEXT,country TEXT,product TEXT,trade_type TEXT)''')
rows=[('Emirates Oil Trading LLC', 'Ahmed Al Mansouri', '+971-4-221-4400', '+971-4-221-4401', '+971-50-100-2200', 'ahmed@emiratesoil.ae', 'info@emiratesoil.ae', 'https://www.emiratesoil.ae', 'JLT Dubai UAE', 'Crude Oil,Refined Petroleum', 'Bulk Export,Tanker Logistics', 'UAE', 'Oil', 'export'), ('Gulf Petroleum Exports', 'Fatima Al Zahra', '+971-2-667-8800', '+971-2-667-8801', '+971-55-200-3300', 'fatima@gulfpetro.ae', 'sales@gulfpetro.ae', 'https://www.gulfpetroexports.ae', 'Abu Dhabi UAE', 'Jet Fuel,Diesel', 'International Shipping,Storage', 'UAE', 'Oil', 'export'), ('Desert Crude Co.', 'Khalid Al Rashid', '+971-4-355-9900', '', '+971-52-300-4400', 'khalid@desertcrude.ae', 'trade@desertcrude.ae', 'https://www.desertcrude.ae', 'Sheikh Zayed Rd Dubai UAE', 'Light Crude Oil,Condensate', 'Trading,Hedging', 'UAE', 'Oil', 'export'), ('ADNOC Distribution Partners', 'Mariam Bint Saeed', '+971-2-605-0000', '+971-2-605-0001', '+971-56-400-5500', 'mariam@adnocdp.ae', 'export@adnocdp.ae', 'https://www.adnocdp.ae', 'Corniche Road Abu Dhabi UAE', 'Premium Gasoline,Aviation Fuel', 'Wholesale Export,Fleet Supply', 'UAE', 'Oil', 'export'), ('Horizon Energy DMCC', 'Omar Bin Khalid', '+971-4-422-7700', '', '+971-54-500-6600', 'omar@horizonenergy.ae', 'info@horizonenergy.ae', 'https://www.horizonenergy.ae', 'Gold Tower JLT Dubai UAE', 'Fuel Oil,Marine Diesel', 'Marine Bunkering,Offshore Supply', 'UAE', 'Oil', 'export'), ('BlueStar Oil and Gas DMCC', 'Hassan Al Thani', '+971-4-511-3300', '', '+971-50-700-8800', 'hassan@bluestarog.ae', 'trade@bluestarog.ae', 'https://www.bluestarog.ae', 'Platinum Tower JLT Dubai UAE', 'Gasoil,Naphtha,LPG', 'Spot Trading,Term Contracts', 'UAE', 'Oil', 'export'), ('Rimal Energy Trading', 'Sara Al Muhairi', '+971-4-234-6600', '+971-4-234-6601', '+971-52-800-9900', 'sara@rimalenergy.ae', 'info@rimalenergy.ae', 'https://www.rimalenergy.ae', 'Business Bay Dubai UAE', 'Crude Oil,Bitumen', 'Pipeline Supply,Storage', 'UAE', 'Oil', 'export'), ('Al Falah Petroleum', 'Noura Al Falah', '+971-3-784-5500', '', '+971-58-600-7700', 'noura@alfalahpetro.ae', 'sales@alfalahpetro.ae', 'https://www.alfalahpetro.ae', 'Al Ain Abu Dhabi UAE', 'Crude Oil,Industrial Oil', 'Export Documentation', 'UAE', 'Oil', 'export'), ('Al Noor Energy Imports', 'Tariq Bin Saeed', '+971-4-300-1100', '', '+971-55-111-2233', 'tariq@alnoorimp.ae', 'info@alnoorimp.ae', 'https://www.alnoorimp.ae', 'DAFZA Dubai UAE', 'Crude Oil,Refined Products', 'Import Clearance,Warehousing', 'UAE', 'Oil', 'import'), ('Gulf Import Solutions', 'Layla Al Ameri', '+971-2-450-2200', '+971-2-450-2201', '+971-56-222-3344', 'layla@gulfimport.ae', 'ops@gulfimport.ae', 'https://www.gulfimportsolutions.ae', 'Khalifa Industrial Zone Abu Dhabi UAE', 'Petroleum Products,Lubricants', 'Bulk Import,Storage', 'UAE', 'Oil', 'import'), ('Saudi Aramco Trading', 'Abdullah Al Ghamdi', '+966-11-872-0000', '', '+966-50-111-9988', 'trading@aramcotrading.sa', 'info@aramcotrading.sa', 'https://www.aramcotrading.com', 'Dhahran Saudi Arabia', 'Arab Light Crude,Naphtha', 'Global Export,Tanker Operations', 'Saudi Arabia', 'Oil', 'export'), ('Red Sea Petroleum Co.', 'Majed Al Otaibi', '+966-12-652-4400', '', '+966-55-222-7766', 'majed@redseapetro.sa', 'sales@redseapetro.sa', 'https://www.redseapetro.sa', 'Jeddah Saudi Arabia', 'Refined Oil,Diesel', 'Port Services,Export Logistics', 'Saudi Arabia', 'Oil', 'export'), ('Karachi Textile Exports', 'Imran Qureshi', '+92-21-3522-1100', '', '+92-300-111-2233', 'imran@ktelexports.pk', 'info@ktelexports.pk', 'https://www.ktelexports.pk', 'SITE Area Karachi Pakistan', 'Cotton Fabric,Denim,Knitwear', 'Cut and Sew,Quality Inspection', 'Pakistan', 'Textile', 'export'), ('Faisalabad Fabrics Co.', 'Ayesha Khan', '+92-41-875-4400', '', '+92-300-222-3344', 'ayesha@ffabrics.pk', 'sales@ffabrics.pk', 'https://www.faisalabadfabrics.pk', 'Faisalabad Pakistan', 'Yarn,Greige Fabric', 'Weaving,Dyeing', 'Pakistan', 'Textile', 'export'), ('Lahore Garments International', 'Usman Malik', '+92-42-357-8800', '', '+92-300-333-4455', 'usman@lahoregarments.pk', 'export@lahoregarments.pk', 'https://www.lahoregarments.pk', 'Lahore Pakistan', 'Readymade Garments,Sportswear', 'CMT,FOB Export', 'Pakistan', 'Textile', 'export'), ('Shenzhen TechExport Co.', 'Li Wei', '+86-755-8899-1100', '', '+86-138-0000-1122', 'liwei@sztechexport.cn', 'info@sztechexport.cn', 'https://www.sztechexport.cn', 'Shenzhen China', 'Smartphones,Laptops,PCBs', 'OEM and ODM,Quality Control', 'China', 'Electronics', 'export'), ('Guangzhou Electronics Ltd', 'Zhang Ming', '+86-20-8811-2200', '', '+86-139-0000-2233', 'zhang@gzelec.cn', 'trade@gzelec.cn', 'https://www.gzelectronics.cn', 'Guangzhou China', 'Consumer Electronics,IoT Devices', 'Product Development,Certification', 'China', 'Electronics', 'export'), ('Mumbai Pharma Exports', 'Rajesh Sharma', '+91-22-6611-0000', '', '+91-98200-11223', 'rajesh@mumbaipharma.in', 'export@mumbaipharma.in', 'https://www.mumbaipharmaexports.in', 'Mumbai India', 'Generic Medicines,APIs,Vaccines', 'GMP Certified,WHO-GMP Export', 'India', 'Pharmaceuticals', 'export'), ('Hyderabad Drug Exports', 'Priya Reddy', '+91-40-6677-8800', '', '+91-98400-22334', 'priya@hydpharma.in', 'sales@hydpharma.in', 'https://www.hyderabaddrugexports.in', 'Hyderabad India', 'Bulk Drugs,Formulations', 'Regulatory Affairs,Export', 'India', 'Pharmaceuticals', 'export')]
cur.executemany('INSERT INTO companies (company_name,contact_person,phone1,phone2,mobile,email1,email2,website,address,products,services,country,product,trade_type) VALUES (?,?,?,?,?,?,?,?,?,?,?,?,?,?)',rows)
conn.commit(); conn.close()
print(f'2/6 Database ready with {len(rows)} companies')

# 3 — AI modules
pathlib.Path('/content/tradefind/ai/nlp_search.py').write_text("import numpy as np, sqlite3, os, pickle\n_model=None; _index=None\nDB=\"/content/tradefind/data/trade.db\"; CACHE=\"/content/tradefind/data/nlp.pkl\"\n\ndef _get_model():\n    global _model\n    if _model is None:\n        from sentence_transformers import SentenceTransformer\n        _model = SentenceTransformer(\"all-MiniLM-L6-v2\")\n    return _model\n\ndef _txt(c):\n    return \" \".join(filter(None,[c.get(\"company_name\",\"\"),c.get(\"product\",\"\"),\n             c.get(\"products\",\"\"),c.get(\"services\",\"\"),c.get(\"country\",\"\"),c.get(\"trade_type\",\"\")]))\n\ndef build_index(force=False):\n    global _index\n    if not force and os.path.exists(CACHE):\n        with open(CACHE,\"rb\") as f: _index=pickle.load(f)\n        print(f\"[NLP] Loaded {len(_index)} companies\"); return\n    print(\"[NLP] Building index...\")\n    conn=sqlite3.connect(DB); conn.row_factory=sqlite3.Row\n    rows=[dict(r) for r in conn.execute(\"SELECT * FROM companies\").fetchall()]; conn.close()\n    vecs=_get_model().encode([_txt(r) for r in rows],show_progress_bar=True,normalize_embeddings=True)\n    _index={r[\"id\"]:(vecs[i],r) for i,r in enumerate(rows)}\n    with open(CACHE,\"wb\") as f: pickle.dump(_index,f)\n    print(f\"[NLP] Index built for {len(_index)} companies\")\n\ndef semantic_search(query,country=\"\",trade_type=\"\",top_k=20,threshold=0.25):\n    if _index is None: build_index()\n    qv=_get_model().encode([query],normalize_embeddings=True)[0]; out=[]\n    for cid,(cv,co) in _index.items():\n        if country and co.get(\"country\",\"\").lower()!=country.lower(): continue\n        if trade_type and co.get(\"trade_type\",\"\").lower()!=trade_type.lower(): continue\n        s=float(np.dot(qv,cv))\n        if s>=threshold: out.append({**co,\"score\":round(s,3)})\n    return sorted(out,key=lambda x:x[\"score\"],reverse=True)[:top_k]\n")
pathlib.Path('/content/tradefind/ai/recommender.py').write_text("import sqlite3, os, pickle\nfrom sklearn.feature_extraction.text import TfidfVectorizer\nfrom sklearn.metrics.pairwise import cosine_similarity\n_vec=None; _mat=None; _cos=None\nDB=\"/content/tradefind/data/trade.db\"; CACHE=\"/content/tradefind/data/tfidf.pkl\"\n\ndef _txt(c):\n    return \" \".join(filter(None,[c.get(\"company_name\",\"\"),c.get(\"product\",\"\"),\n             c.get(\"products\",\"\"),c.get(\"services\",\"\"),c.get(\"trade_type\",\"\")])).lower()\n\ndef build_index(force=False):\n    global _vec,_mat,_cos\n    if not force and os.path.exists(CACHE):\n        with open(CACHE,\"rb\") as f: _vec,_mat,_cos=pickle.load(f)\n        print(f\"[Rec] Loaded {len(_cos)} companies\"); return\n    print(\"[Rec] Building TF-IDF index...\")\n    conn=sqlite3.connect(DB); conn.row_factory=sqlite3.Row\n    _cos=[dict(r) for r in conn.execute(\"SELECT * FROM companies\").fetchall()]; conn.close()\n    _vec=TfidfVectorizer(ngram_range=(1,2),min_df=1,stop_words=\"english\")\n    _mat=_vec.fit_transform([_txt(c) for c in _cos])\n    with open(CACHE,\"wb\") as f: pickle.dump((_vec,_mat,_cos),f)\n    print(f\"[Rec] Index built for {len(_cos)} companies\")\n\ndef get_similar(company_id,top_k=4):\n    if _cos is None: build_index()\n    idx=next((i for i,c in enumerate(_cos) if c[\"id\"]==company_id),None)\n    if idx is None: return []\n    scores=cosine_similarity(_mat[idx],_mat).flatten()\n    ranked=sorted([(i,s) for i,s in enumerate(scores) if i!=idx],key=lambda x:x[1],reverse=True)\n    return [{**_cos[i],\"similarity\":round(float(s),3)} for i,s in ranked[:top_k] if s>0.05]\n")
pathlib.Path('/content/tradefind/ai/chatbot.py').write_text("import json, sqlite3, os, re\nDB=\"/content/tradefind/data/trade.db\"\nSYS=\"\"\"You are a trade directory assistant. Extract filters as JSON only.\nCountries: UAE, Saudi Arabia, Pakistan, China, India\nProducts: Oil, Textile, Electronics, Pharmaceuticals\nTrade types: export, import\nReturn ONLY this JSON with no extra text:\n{\"country\":\"\",\"product\":\"\",\"trade_type\":\"\",\"has_email\":false,\"has_website\":false,\"explanation\":\"one sentence\"}\"\"\"\n\ndef _client():\n    from groq import Groq\n    k=os.environ.get(\"GROQ_API_KEY\",\"\")\n    if not k: raise ValueError(\"GROQ_API_KEY not set!\")\n    return Groq(api_key=k)\n\ndef _filters(msg):\n    r=_client().chat.completions.create(\n        model=\"llama-3.3-70b-versatile\",\n        messages=[{\"role\":\"system\",\"content\":SYS},{\"role\":\"user\",\"content\":msg}],\n        temperature=0.1, max_tokens=200)\n    raw=re.sub(r\"```(?:json)?\",\"\",r.choices[0].message.content).strip()\n    try: return json.loads(raw)\n    except: return {\"country\":\"\",\"product\":\"\",\"trade_type\":\"\",\"has_email\":False,\"has_website\":False,\"explanation\":\"Could not parse.\"}\n\ndef _db(f):\n    conn=sqlite3.connect(DB); conn.row_factory=sqlite3.Row\n    q,p=\"SELECT * FROM companies WHERE 1=1\",[]\n    if f.get(\"country\"):    q+=\" AND LOWER(country)=LOWER(?)\";    p.append(f[\"country\"])\n    if f.get(\"product\"):    q+=\" AND LOWER(product)=LOWER(?)\";    p.append(f[\"product\"])\n    if f.get(\"trade_type\"): q+=\" AND LOWER(trade_type)=LOWER(?)\"  ;p.append(f[\"trade_type\"])\n    if f.get(\"has_email\"):  q+=\" AND email1 IS NOT NULL AND email1 != \\'\\'\"\n    if f.get(\"has_website\"):q+=\" AND website IS NOT NULL AND website != \\'\\'\"\n    rows=conn.execute(q,p).fetchall(); conn.close()\n    return [dict(r) for r in rows]\n\ndef chat(message, history=None):\n    try:\n        f=_filters(message); exp=f.pop(\"explanation\",\"\"); cos=_db(f); n=len(cos)\n        parts=([f\"{v} {k}\" for k,v in f.items() if v and k not in(\"has_email\",\"has_website\")]\n               +([\"with email\"] if f.get(\"has_email\") else [])\n               +([\"with website\"] if f.get(\"has_website\") else []))\n        fs=\", \".join(parts) if parts else \"all categories\"\n        reply=f\"Found **{n}** companies for **{fs}**. {exp}\" if n else f\"No results for **{fs}**. Try broader filters.\"\n        return {\"reply\":reply,\"companies\":cos,\"filters\":f,\"error\":None}\n    except ValueError as e: return {\"reply\":str(e),\"companies\":[],\"filters\":{},\"error\":\"key_missing\"}\n    except Exception as e:  return {\"reply\":f\"Error: {e}\",\"companies\":[],\"filters\":{},\"error\":str(e)}\n")
print('3/6 AI modules ready')

# 4 — HTML templates with CSS and JS already baked in
pathlib.Path('/content/tradefind/templates/index.html').write_text("<!DOCTYPE html>\n<html lang=\"en\">\n<head>\n<meta charset=\"UTF-8\">\n<meta name=\"viewport\" content=\"width=device-width, initial-scale=1.0\">\n<title>TradeFind - AI Import Export Directory</title>\n<link href=\"https://fonts.googleapis.com/css2?family=Syne:wght@700;800&family=DM+Sans:wght@300;400;500&display=swap\" rel=\"stylesheet\">\n<style>/* \u2500\u2500 Reset & Base \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500 */\n*, *::before, *::after { box-sizing: border-box; margin: 0; padding: 0; }\n:root {\n  --navy:    #0d1f3c;\n  --navy2:   #1b3a6b;\n  --blue:    #2563eb;\n  --blue-lt: #dbeafe;\n  --accent:  #f59e0b;\n  --green:   #10b981;\n  --red:     #ef4444;\n  --gray1:   #f8fafc;\n  --gray2:   #f1f5f9;\n  --gray3:   #e2e8f0;\n  --gray5:   #64748b;\n  --gray7:   #334155;\n  --gray9:   #0f172a;\n  --white:   #ffffff;\n  --font-head: 'Syne', sans-serif;\n  --font-body: 'DM Sans', sans-serif;\n  --radius:  12px;\n  --shadow:  0 4px 24px rgba(13,31,60,.10);\n  --shadow2: 0 8px 40px rgba(13,31,60,.16);\n}\n\nbody {\n  font-family: var(--font-body);\n  color: var(--gray9);\n  background: #f0f4fa;\n  min-height: 100vh;\n  line-height: 1.6;\n  position: relative;\n}\n\n/* \u2500\u2500 Background grid \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500 */\n.bg-grid {\n  position: fixed; inset: 0; z-index: 0; pointer-events: none;\n  background-image:\n    linear-gradient(rgba(37,99,235,.04) 1px, transparent 1px),\n    linear-gradient(90deg, rgba(37,99,235,.04) 1px, transparent 1px);\n  background-size: 48px 48px;\n}\n\n.container { max-width: 1200px; margin: 0 auto; padding: 0 24px; position: relative; z-index: 1; }\n\n/* \u2500\u2500 Header \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500 */\n.site-header {\n  background: var(--navy);\n  padding: 0 0;\n  position: sticky; top: 0; z-index: 100;\n  border-bottom: 1px solid rgba(255,255,255,.06);\n}\n.site-header .container { display: flex; align-items: center; justify-content: space-between; height: 60px; }\n\n.logo { display: flex; align-items: center; gap: 10px; text-decoration: none; color: var(--white); }\n.logo-mark { color: var(--accent); font-size: 18px; line-height: 1; }\n.logo-text  { font-family: var(--font-head); font-size: 20px; color: var(--white); }\n.logo-text strong { color: var(--accent); }\n\nnav a { color: rgba(255,255,255,.7); text-decoration: none; font-size: 14px; margin-left: 24px;\n        transition: color .2s; }\nnav a:hover { color: var(--white); }\n\n/* \u2500\u2500 Hero \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500 */\n.hero { padding: 64px 0 48px; }\n.hero-badge {\n  display: inline-block; background: var(--blue-lt); color: var(--blue);\n  font-size: 12px; font-weight: 500; letter-spacing: .08em; text-transform: uppercase;\n  padding: 4px 12px; border-radius: 20px; margin-bottom: 20px;\n}\n.hero h1 {\n  font-family: var(--font-head); font-size: clamp(36px, 5vw, 60px);\n  font-weight: 800; line-height: 1.12; color: var(--navy); margin-bottom: 16px;\n}\n.hero h1 em { font-style: normal; color: var(--blue); }\n.hero-sub { font-size: 18px; color: var(--gray5); max-width: 540px; margin-bottom: 40px; font-weight: 300; }\n\n/* \u2500\u2500 Search Card \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500 */\n.search-card {\n  background: var(--white);\n  border-radius: 20px;\n  box-shadow: var(--shadow2);\n  padding: 36px 40px;\n  border: 1px solid var(--gray3);\n}\n\n.form-grid { display: grid; grid-template-columns: 1fr 1fr 1fr; gap: 24px; margin-bottom: 28px; }\n@media (max-width: 768px) { .form-grid { grid-template-columns: 1fr; } }\n\n.form-group label {\n  display: block; font-size: 13px; font-weight: 500; color: var(--gray7);\n  margin-bottom: 8px; text-transform: uppercase; letter-spacing: .06em;\n}\n\n.select-wrap { position: relative; }\n.select-wrap::after {\n  content: '\u25be'; position: absolute; right: 14px; top: 50%;\n  transform: translateY(-50%); color: var(--gray5); pointer-events: none; font-size: 13px;\n}\nselect {\n  width: 100%; padding: 12px 40px 12px 16px;\n  border: 1.5px solid var(--gray3); border-radius: var(--radius);\n  font-family: var(--font-body); font-size: 15px; color: var(--gray9);\n  background: var(--gray1); appearance: none; cursor: pointer;\n  transition: border-color .2s, box-shadow .2s;\n}\nselect:focus { outline: none; border-color: var(--blue); box-shadow: 0 0 0 3px rgba(37,99,235,.12); }\n\n/* \u2500\u2500 Radio Toggle \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500 */\n.radio-group { display: flex; gap: 12px; }\n.radio-btn { flex: 1; }\n.radio-btn input { display: none; }\n.radio-btn span {\n  display: flex; align-items: center; justify-content: center;\n  padding: 12px; border: 1.5px solid var(--gray3); border-radius: var(--radius);\n  font-size: 15px; font-weight: 500; cursor: pointer; background: var(--gray1);\n  transition: all .2s;\n}\n.radio-btn input:checked + span {\n  background: var(--navy2); color: var(--white);\n  border-color: var(--navy2);\n}\n.radio-btn:hover span { border-color: var(--blue); }\n\n/* \u2500\u2500 Search Button \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500 */\n.btn-search {\n  display: inline-flex; align-items: center; gap: 10px;\n  background: var(--blue); color: var(--white);\n  border: none; border-radius: var(--radius);\n  font-family: var(--font-head); font-size: 16px; font-weight: 600;\n  padding: 14px 36px; cursor: pointer;\n  transition: background .2s, transform .15s, box-shadow .2s;\n  box-shadow: 0 4px 16px rgba(37,99,235,.35);\n}\n.btn-search:hover { background: #1d4ed8; transform: translateY(-1px); box-shadow: 0 6px 24px rgba(37,99,235,.45); }\n.btn-search:active { transform: translateY(0); }\n\n/* \u2500\u2500 Loader \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500 */\n.loader { text-align: center; padding: 60px 0; display: flex; flex-direction: column; align-items: center; gap: 16px; }\n.spinner {\n  width: 40px; height: 40px;\n  border: 3px solid var(--gray3);\n  border-top-color: var(--blue);\n  border-radius: 50%;\n  animation: spin .7s linear infinite;\n}\n@keyframes spin { to { transform: rotate(360deg); } }\n.loader p { color: var(--gray5); font-size: 14px; }\n\n/* \u2500\u2500 Results Section \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500 */\n.results-section { padding: 48px 0 80px; }\n.results-header { display: flex; align-items: center; justify-content: space-between; margin-bottom: 24px; flex-wrap: wrap; gap: 16px; }\n.results-header h2 { font-family: var(--font-head); font-size: 28px; font-weight: 700; color: var(--navy); }\n.results-meta { font-size: 14px; color: var(--gray5); margin-top: 2px; }\n\n.btn-export {\n  display: inline-flex; align-items: center; gap: 8px;\n  background: var(--green); color: var(--white);\n  border: none; border-radius: var(--radius);\n  font-family: var(--font-body); font-size: 14px; font-weight: 500;\n  padding: 10px 22px; cursor: pointer;\n  transition: background .2s, transform .15s;\n  text-decoration: none;\n}\n.btn-export:hover { background: #059669; transform: translateY(-1px); }\n\n/* \u2500\u2500 Table \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500 */\n.table-wrap { overflow-x: auto; border-radius: 16px; box-shadow: var(--shadow); }\n.results-table {\n  width: 100%; border-collapse: collapse;\n  background: var(--white); font-size: 14px;\n  border-radius: 16px; overflow: hidden;\n}\n.results-table thead tr { background: var(--navy); }\n.results-table th {\n  padding: 14px 16px; text-align: left;\n  color: rgba(255,255,255,.85); font-size: 12px;\n  font-weight: 600; letter-spacing: .06em; text-transform: uppercase;\n  white-space: nowrap;\n}\n.results-table tbody tr { border-bottom: 1px solid var(--gray3); transition: background .15s; }\n.results-table tbody tr:last-child { border-bottom: none; }\n.results-table tbody tr:hover { background: var(--blue-lt); }\n.results-table td { padding: 13px 16px; color: var(--gray7); vertical-align: middle; }\n.results-table td:first-child { font-weight: 600; color: var(--gray5); width: 48px; }\n.results-table .company-name { font-weight: 600; color: var(--navy); white-space: nowrap; }\na.detail-link {\n  display: inline-flex; align-items: center; gap: 4px;\n  color: var(--blue); font-weight: 500; text-decoration: none; font-size: 13px;\n  padding: 5px 12px; border: 1px solid var(--blue-lt); border-radius: 20px;\n  transition: background .15s;\n}\na.detail-link:hover { background: var(--blue-lt); }\n\n/* \u2500\u2500 No Results \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500 */\n.no-results {\n  background: var(--white); border-radius: 16px; text-align: center;\n  padding: 60px 40px; box-shadow: var(--shadow);\n}\n.no-results-icon { font-size: 36px; margin-bottom: 16px; }\n.no-results h3 { font-family: var(--font-head); font-size: 22px; color: var(--navy); margin-bottom: 8px; }\n.no-results p { color: var(--gray5); }\n\n/* \u2500\u2500 Detail Page \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500 */\n.detail-hero { padding: 56px 0 32px; }\n.detail-badges { display: flex; gap: 10px; margin-bottom: 16px; flex-wrap: wrap; }\n.badge {\n  display: inline-block; padding: 4px 14px; border-radius: 20px;\n  font-size: 12px; font-weight: 600; letter-spacing: .06em; text-transform: uppercase;\n}\n.badge-type.export { background: #dcfce7; color: #166534; }\n.badge-type.import { background: #dbeafe; color: #1e40af; }\n.badge-product  { background: #fef3c7; color: #92400e; }\n.badge-country  { background: var(--gray2); color: var(--gray7); }\n\n.detail-title { font-family: var(--font-head); font-size: clamp(28px,4vw,46px); font-weight: 800; color: var(--navy); margin-bottom: 14px; }\n\n.website-link {\n  display: inline-flex; align-items: center; gap: 6px;\n  color: var(--blue); text-decoration: none; font-size: 14px; font-weight: 500;\n  padding: 6px 16px; border: 1.5px solid var(--blue-lt); border-radius: 20px;\n  transition: background .15s;\n}\n.website-link:hover { background: var(--blue-lt); }\n\n.detail-body { padding: 0 0 80px; }\n.detail-grid { display: grid; grid-template-columns: 1fr 1fr; gap: 20px; margin-bottom: 32px; }\n@media (max-width: 768px) { .detail-grid { grid-template-columns: 1fr; } }\n.full-width { grid-column: 1 / -1; }\n\n.detail-card {\n  background: var(--white); border-radius: 16px; border: 1px solid var(--gray3);\n  box-shadow: var(--shadow); overflow: hidden;\n}\n.card-header {\n  display: flex; align-items: center; gap: 12px;\n  padding: 18px 24px; border-bottom: 1px solid var(--gray3);\n  background: var(--gray1);\n}\n.card-icon { font-size: 18px; }\n.card-header h3 { font-family: var(--font-head); font-size: 16px; font-weight: 700; color: var(--navy); }\n.card-body { padding: 20px 24px; }\n\n.info-row { display: flex; align-items: baseline; justify-content: space-between; gap: 16px; padding: 10px 0; border-bottom: 1px solid var(--gray2); }\n.info-row:last-child { border-bottom: none; }\n.info-label { font-size: 13px; color: var(--gray5); font-weight: 500; white-space: nowrap; }\n.info-value { font-size: 14px; color: var(--gray9); font-weight: 500; text-align: right; }\n.info-value a { color: var(--blue); text-decoration: none; }\n.info-value a:hover { text-decoration: underline; }\n\n.address-text { font-size: 15px; color: var(--gray7); line-height: 1.7; margin-bottom: 16px; }\n.map-link { color: var(--blue); font-size: 13px; font-weight: 500; text-decoration: none; }\n.map-link:hover { text-decoration: underline; }\n\n.two-col { display: grid; grid-template-columns: 1fr 1fr; gap: 24px; }\n@media (max-width: 640px) { .two-col { grid-template-columns: 1fr; } }\n.two-col h4 { font-family: var(--font-head); font-size: 14px; font-weight: 700; color: var(--navy); margin-bottom: 12px; text-transform: uppercase; letter-spacing: .06em; }\n\n.tag-list { display: flex; flex-wrap: wrap; gap: 8px; }\n.tag {\n  background: var(--blue-lt); color: #1e40af;\n  padding: 5px 14px; border-radius: 20px; font-size: 13px; font-weight: 500;\n}\n.tag-service { background: #f0fdf4; color: #166534; }\n\n.detail-actions { display: flex; gap: 16px; flex-wrap: wrap; }\n.btn-back {\n  display: inline-flex; align-items: center; gap: 6px;\n  padding: 12px 24px; border-radius: var(--radius);\n  border: 1.5px solid var(--gray3); color: var(--gray7);\n  font-family: var(--font-body); font-size: 14px; font-weight: 500;\n  text-decoration: none; transition: border-color .2s, background .2s;\n}\n.btn-back:hover { border-color: var(--blue); color: var(--blue); background: var(--blue-lt); }\n.btn-visit {\n  display: inline-flex; align-items: center; gap: 6px;\n  padding: 12px 24px; border-radius: var(--radius);\n  background: var(--blue); color: var(--white);\n  font-family: var(--font-body); font-size: 14px; font-weight: 500;\n  text-decoration: none; transition: background .2s, transform .15s;\n}\n.btn-visit:hover { background: #1d4ed8; transform: translateY(-1px); }\n\n/* \u2500\u2500 AI Badge \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500 */\n.ai-badge {\n  display: inline-flex; align-items: center; gap: 8px;\n  background: linear-gradient(135deg, #1e1b4b, #1e3a5f);\n  color: #a5b4fc; font-size: 13px; font-weight: 500;\n  padding: 8px 16px; border-radius: 20px; margin-bottom: 32px;\n  border: 1px solid rgba(165,180,252,.25);\n}\n.ai-dot {\n  width: 8px; height: 8px; border-radius: 50%;\n  background: #818cf8;\n  box-shadow: 0 0 0 3px rgba(129,140,248,.25);\n  animation: pulse 2s ease-in-out infinite;\n  flex-shrink: 0;\n}\n@keyframes pulse { 0%,100%{box-shadow:0 0 0 3px rgba(129,140,248,.25)} 50%{box-shadow:0 0 0 6px rgba(129,140,248,.10)} }\n\n/* \u2500\u2500 Score bar \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500 */\n.score-bar-wrap {\n  display: flex; align-items: center; gap: 6px;\n  width: 90px; cursor: help;\n}\n.score-bar {\n  height: 6px; border-radius: 3px;\n  transition: width .4s ease;\n  flex-shrink: 0;\n}\n.score-label { font-size: 12px; font-weight: 600; white-space: nowrap; }\n\n/* \u2500\u2500 Footer \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500 */\n.site-footer {\n  background: var(--navy); color: rgba(255,255,255,.45);\n  padding: 24px 0; text-align: center; font-size: 13px;\n}\n\n/* \u2500\u2500 AI Badge \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500 */\n.ai-badge {\n  background: linear-gradient(90deg,#7f77dd,#2563eb);\n  color:#fff; font-size:11px; font-weight:600; letter-spacing:.06em;\n  padding:3px 10px; border-radius:20px; margin-left:10px; vertical-align:middle;\n}\n.ai-label {\n  background:var(--blue-lt); color:var(--blue); font-size:11px; font-weight:600;\n  padding:2px 10px; border-radius:20px; margin-left:8px; letter-spacing:.04em;\n}\n\n/* \u2500\u2500 Search Mode Toggle \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500 */\n.search-mode-toggle { display:flex; gap:8px; margin-bottom:16px; }\n.mode-btn {\n  padding:8px 20px; border-radius:20px; border:1.5px solid var(--gray3);\n  font-family:var(--font-body); font-size:14px; font-weight:500; cursor:pointer;\n  background:var(--white); color:var(--gray5); transition:all .2s;\n}\n.mode-btn.active { background:var(--navy); color:var(--white); border-color:var(--navy); }\n.mode-btn:hover:not(.active) { border-color:var(--blue); color:var(--blue); }\n\n/* \u2500\u2500 AI Search Card \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500 */\n.ai-card { border:1.5px solid #c4b5fd; background:#faf9ff; }\n.ai-card-header { display:flex; align-items:flex-start; gap:14px; margin-bottom:20px; }\n.ai-icon-lg { font-size:22px; color:#7f77dd; line-height:1; margin-top:2px; }\n.ai-card-title { font-family:var(--font-head); font-size:17px; font-weight:700; color:var(--navy); }\n.ai-card-sub { font-size:13px; color:var(--gray5); margin-top:3px; }\n.ai-form-row { display:flex; gap:12px; flex-wrap:wrap; align-items:stretch; }\n.ai-input {\n  flex:1; min-width:220px; padding:12px 16px;\n  border:1.5px solid var(--gray3); border-radius:var(--radius);\n  font-family:var(--font-body); font-size:15px; color:var(--gray9); background:var(--white);\n  transition:border-color .2s, box-shadow .2s;\n}\n.ai-input:focus { outline:none; border-color:#7f77dd; box-shadow:0 0 0 3px rgba(127,119,221,.12); }\n.ai-select-wrap { min-width:140px; }\n\n/* \u2500\u2500 Score column \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500 */\n.score-pill {\n  display:inline-block; padding:3px 10px; border-radius:20px;\n  font-size:12px; font-weight:600; background:#ede9fe; color:#4c1d95;\n}\n\n/* \u2500\u2500 Chatbot FAB \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500 */\n.chat-fab {\n  position:fixed; bottom:28px; right:28px; z-index:200;\n  display:flex; align-items:center; gap:8px;\n  background:var(--navy); color:var(--white);\n  border:none; border-radius:30px; padding:12px 22px;\n  font-family:var(--font-body); font-size:14px; font-weight:500; cursor:pointer;\n  box-shadow:0 4px 20px rgba(13,31,60,.3); transition:transform .2s, box-shadow .2s;\n}\n.chat-fab:hover { transform:translateY(-2px); box-shadow:0 6px 28px rgba(13,31,60,.4); }\n\n/* \u2500\u2500 Chatbot Panel \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500 */\n.chat-panel {\n  position:fixed; bottom:90px; right:28px; z-index:300;\n  width:380px; max-height:520px;\n  background:var(--white); border-radius:20px;\n  box-shadow:0 12px 48px rgba(13,31,60,.2);\n  border:1px solid var(--gray3);\n  display:none; flex-direction:column; overflow:hidden;\n}\n.chat-panel.open { display:flex; }\n.chat-header {\n  display:flex; align-items:center; justify-content:space-between;\n  padding:16px 20px; background:var(--navy); color:var(--white);\n}\n.chat-title { display:flex; align-items:center; gap:8px; font-family:var(--font-head); font-size:15px; font-weight:700; }\n.ai-icon { color:#a78bfa; font-size:16px; }\n.chat-close { background:none; border:none; color:rgba(255,255,255,.6); font-size:18px; cursor:pointer; line-height:1; }\n.chat-close:hover { color:var(--white); }\n.chat-messages { flex:1; overflow-y:auto; padding:16px; display:flex; flex-direction:column; gap:12px; }\n.chat-msg { display:flex; }\n.chat-msg.assistant .msg-bubble {\n  background:var(--gray1); border:1px solid var(--gray3); border-radius:4px 16px 16px 16px;\n  padding:12px 14px; font-size:14px; color:var(--gray7); line-height:1.6; max-width:90%;\n}\n.chat-msg.user { justify-content:flex-end; }\n.chat-msg.user .msg-bubble {\n  background:var(--navy); color:var(--white);\n  border-radius:16px 4px 16px 16px;\n  padding:10px 14px; font-size:14px; max-width:85%;\n}\n.chat-companies { margin-top:10px; display:flex; flex-direction:column; gap:6px; }\n.chat-company-link {\n  display:block; padding:8px 12px; border-radius:8px;\n  background:var(--white); border:1px solid var(--gray3);\n  font-size:13px; color:var(--navy); text-decoration:none; font-weight:500;\n  transition:background .15s;\n}\n.chat-company-link:hover { background:var(--blue-lt); }\n.chat-input-row {\n  display:flex; gap:8px; padding:12px 16px;\n  border-top:1px solid var(--gray3);\n}\n.chat-input-row input {\n  flex:1; padding:10px 14px; border:1.5px solid var(--gray3);\n  border-radius:20px; font-family:var(--font-body); font-size:14px;\n  transition:border-color .2s;\n}\n.chat-input-row input:focus { outline:none; border-color:var(--blue); }\n.btn-send {\n  width:40px; height:40px; border-radius:50%; background:var(--blue); color:var(--white);\n  border:none; cursor:pointer; display:flex; align-items:center; justify-content:center;\n  transition:background .2s;\n}\n.btn-send:hover { background:#1d4ed8; }\n.typing-dots span {\n  display:inline-block; width:6px; height:6px; border-radius:50%;\n  background:var(--gray5); margin:0 2px;\n  animation:bounce .9s infinite; \n}\n.typing-dots span:nth-child(2) { animation-delay:.15s; }\n.typing-dots span:nth-child(3) { animation-delay:.3s; }\n@keyframes bounce { 0%,80%,100%{transform:translateY(0)} 40%{transform:translateY(-6px)} }\n\n/* \u2500\u2500 Similar Companies \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500 */\n.similar-grid { display:grid; grid-template-columns:1fr 1fr; gap:12px; }\n@media(max-width:640px){ .similar-grid{grid-template-columns:1fr;} }\n.similar-card {\n  display:block; padding:14px 16px; border-radius:12px;\n  border:1.5px solid var(--gray3); text-decoration:none;\n  transition:border-color .2s, background .2s;\n}\n.similar-card:hover { border-color:var(--blue); background:var(--blue-lt); }\n.similar-name { font-weight:600; color:var(--navy); font-size:14px; margin-bottom:4px; }\n.similar-meta { font-size:12px; color:var(--gray5); margin-bottom:10px; }\n.similar-score { display:flex; align-items:center; gap:8px; }\n.score-bar { flex:1; height:5px; background:var(--gray3); border-radius:3px; overflow:hidden; }\n.score-fill { height:100%; background:var(--blue); border-radius:3px; }\n.similar-score span { font-size:11px; color:var(--gray5); font-weight:500; white-space:nowrap; }\n</style>\n</head>\n<body>\n<div class=\"bg-grid\"></div>\n<header class=\"site-header\">\n  <div class=\"container\">\n    <div class=\"logo\"><span class=\"logo-mark\">TF</span><span class=\"logo-text\">Trade<strong>Find</strong></span>{% if ai_enabled %}<span class=\"ai-badge\">AI Powered</span>{% endif %}</div>\n    <nav><a href=\"/\">Search</a>{% if ai_enabled %}<a href=\"#\" onclick=\"toggleChat(true);return false;\">AI Chat</a>{% endif %}</nav>\n  </div>\n</header>\n<main>\n  <section class=\"hero\"><div class=\"container\">\n    <div class=\"hero-badge\">Global Trade Directory</div>\n    <h1>Find <em>Importers</em> and <br>Exporters Worldwide</h1>\n    <p class=\"hero-sub\">Search verified trade contacts. Download company lists in Excel instantly.</p>\n    {% if ai_enabled %}\n    <div class=\"search-mode-toggle\">\n      <button class=\"mode-btn active\" id=\"btnClassic\" onclick=\"setMode('classic')\">Classic Search</button>\n      <button class=\"mode-btn\" id=\"btnAI\" onclick=\"setMode('ai')\">AI Semantic Search</button>\n    </div>{% endif %}\n    <div class=\"search-card\" id=\"classicCard\">\n      <form id=\"searchForm\">\n        <div class=\"form-grid\">\n          <div class=\"form-group\"><label>Country</label><div class=\"select-wrap\"><select name=\"country\" id=\"country\" required><option value=\"\">Select country</option>{% for c in countries %}<option value=\"{{ c }}\">{{ c }}</option>{% endfor %}</select></div></div>\n          <div class=\"form-group\"><label>Product</label><div class=\"select-wrap\"><select name=\"product\" id=\"product\" required><option value=\"\">Select product</option>{% for p in products %}<option value=\"{{ p }}\">{{ p }}</option>{% endfor %}</select></div></div>\n          <div class=\"form-group\"><label>Trade Type</label><div class=\"radio-group\"><label class=\"radio-btn\"><input type=\"radio\" name=\"trade_type\" value=\"export\" checked><span>Exporter</span></label><label class=\"radio-btn\"><input type=\"radio\" name=\"trade_type\" value=\"import\"><span>Importer</span></label></div></div>\n        </div>\n        <button type=\"submit\" class=\"btn-search\">Search Companies</button>\n      </form>\n    </div>\n    {% if ai_enabled %}\n    <div class=\"search-card ai-card\" id=\"aiCard\" style=\"display:none\">\n      <div class=\"ai-card-header\"><span class=\"ai-icon-lg\">AI</span><div><div class=\"ai-card-title\">AI Semantic Search</div><div class=\"ai-card-sub\">Type anything - petroleum finds oil, garments finds textile</div></div></div>\n      <div class=\"ai-form-row\">\n        <input type=\"text\" id=\"aiQuery\" placeholder=\"e.g. crude oil supplier Gulf region\" class=\"ai-input\" onkeydown=\"if(event.key==='Enter') runAISearch()\"/>\n        <div class=\"select-wrap ai-select-wrap\"><select id=\"aiCountry\"><option value=\"\">All countries</option>{% for c in countries %}<option value=\"{{ c }}\">{{ c }}</option>{% endfor %}</select></div>\n        <div class=\"radio-group\"><label class=\"radio-btn\"><input type=\"radio\" name=\"ai_trade\" value=\"\" checked><span>Any</span></label><label class=\"radio-btn\"><input type=\"radio\" name=\"ai_trade\" value=\"export\"><span>Export</span></label><label class=\"radio-btn\"><input type=\"radio\" name=\"ai_trade\" value=\"import\"><span>Import</span></label></div>\n        <button class=\"btn-search\" onclick=\"runAISearch()\">Search</button>\n      </div>\n    </div>{% endif %}\n  </div></section>\n  <section class=\"results-section\" id=\"resultsSection\" style=\"display:none\"><div class=\"container\">\n    <div class=\"results-header\"><div><h2 id=\"resultsTitle\">Results</h2><p id=\"resultsCount\" class=\"results-meta\"></p></div><button class=\"btn-export\" id=\"exportBtn\">Download Excel</button></div>\n    <div class=\"table-wrap\"><table class=\"results-table\"><thead><tr><th>SR#</th><th>Company Name</th><th>Contact Person</th><th>Phone 1</th><th>Phone 2</th><th>Mobile</th><th>Email 1</th><th>Email 2</th><th id=\"scoreHeader\" style=\"display:none\">AI Score</th><th>Detail</th></tr></thead><tbody id=\"resultsBody\"></tbody></table></div>\n    <div id=\"noResults\" class=\"no-results\" style=\"display:none\"><h3>No companies found</h3><p>Try different filters.</p></div>\n  </div></section>\n  <div class=\"loader\" id=\"loader\" style=\"display:none\"><div class=\"spinner\"></div><p id=\"loaderText\">Searching...</p></div>\n</main>\n{% if ai_enabled %}\n<div class=\"chat-panel\" id=\"chatPanel\">\n  <div class=\"chat-header\"><div class=\"chat-title\">AI Trade Assistant</div><button class=\"chat-close\" onclick=\"toggleChat(false)\">X</button></div>\n  <div class=\"chat-messages\" id=\"chatMessages\"><div class=\"chat-msg assistant\"><div class=\"msg-bubble\">Hi! Try asking:<br><br>Find oil exporters in UAE with email<br>Pakistani textile companies with a website</div></div></div>\n  <div class=\"chat-input-row\"><input type=\"text\" id=\"chatInput\" placeholder=\"Ask anything...\" onkeydown=\"if(event.key==='Enter') sendChat()\"/><button class=\"btn-send\" onclick=\"sendChat()\">Send</button></div>\n</div>\n<button class=\"chat-fab\" onclick=\"toggleChat(true)\">AI Chat</button>\n{% endif %}\n<footer class=\"site-footer\"><div class=\"container\"><p>2025 TradeFind - AI Import Export Directory</p></div></footer>\n<script>const loader     = document.getElementById('loader');\nconst loaderText = document.getElementById('loaderText');\nconst resultsSection = document.getElementById('resultsSection');\nconst resultsBody    = document.getElementById('resultsBody');\nconst resultsTitle   = document.getElementById('resultsTitle');\nconst resultsCount   = document.getElementById('resultsCount');\nconst noResults      = document.getElementById('noResults');\nconst tableWrap      = document.querySelector('.table-wrap');\nconst scoreHeader    = document.getElementById('scoreHeader');\nconst exportBtn      = document.getElementById('exportBtn');\n\nlet lastParams  = {};\nlet currentMode = 'classic';\n\n// \u2500\u2500 Search mode toggle \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\nfunction setMode(mode) {\n  currentMode = mode;\n  const classicCard = document.getElementById('classicCard');\n  const aiCard      = document.getElementById('aiCard');\n  const btnClassic  = document.getElementById('btnClassic');\n  const btnAI       = document.getElementById('btnAI');\n  if (!classicCard || !aiCard) return;\n  if (mode === 'classic') {\n    classicCard.style.display = 'block';\n    aiCard.style.display      = 'none';\n    btnClassic.classList.add('active');\n    btnAI.classList.remove('active');\n  } else {\n    classicCard.style.display = 'none';\n    aiCard.style.display      = 'block';\n    btnAI.classList.add('active');\n    btnClassic.classList.remove('active');\n    setTimeout(() => document.getElementById('aiQuery')?.focus(), 100);\n  }\n}\n\n// \u2500\u2500 Classic search \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\nconst form = document.getElementById('searchForm');\nif (form) {\n  form.addEventListener('submit', async (e) => {\n    e.preventDefault();\n    const country    = document.getElementById('country').value;\n    const product    = document.getElementById('product').value;\n    const trade_type = document.querySelector('input[name=\"trade_type\"]:checked').value;\n    if (!country || !product) { alert('Please select both a Country and a Product.'); return; }\n    lastParams = { country, product, trade_type };\n    showLoader('Searching directory\u2026');\n    try {\n      const res  = await fetch('/search?' + new URLSearchParams(lastParams));\n      const data = await res.json();\n      hideLoader();\n      const typeLabel = trade_type === 'export' ? 'Exporters' : 'Importers';\n      showResults(data, `${product} ${typeLabel} \u2014 ${country}`, false);\n    } catch (err) {\n      hideLoader();\n      alert('Error fetching results. Please try again.');\n    }\n  });\n}\n\n// \u2500\u2500 AI semantic search \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\nasync function runAISearch() {\n  const q          = document.getElementById('aiQuery')?.value.trim();\n  const country    = document.getElementById('aiCountry')?.value;\n  const trade_type = document.querySelector('input[name=\"ai_trade\"]:checked')?.value || '';\n  if (!q) { alert('Please enter a search query.'); return; }\n  lastParams = { country, product: '', trade_type };\n  showLoader('AI is searching semantically\u2026');\n  try {\n    const qs  = new URLSearchParams({ q, country, trade_type });\n    const res = await fetch('/ai/search?' + qs);\n    const data = await res.json();\n    hideLoader();\n    if (data.error) { alert('AI Search error: ' + data.error); return; }\n    showResults(data, `AI results for \"${q}\"`, true);\n  } catch (err) {\n    hideLoader();\n    alert('AI search failed. Please try again.');\n  }\n}\n\n// \u2500\u2500 Render results \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\nfunction showResults(data, title, showScore) {\n  resultsSection.style.display = 'block';\n  resultsTitle.textContent     = title;\n  resultsCount.textContent     = `${data.length} compan${data.length === 1 ? 'y' : 'ies'} found`;\n  if (scoreHeader) scoreHeader.style.display = showScore ? '' : 'none';\n  if (data.length === 0) {\n    tableWrap.style.display   = 'none';\n    noResults.style.display   = 'block';\n  } else {\n    tableWrap.style.display   = 'block';\n    noResults.style.display   = 'none';\n    renderTable(data, showScore);\n  }\n  resultsSection.scrollIntoView({ behavior: 'smooth', block: 'start' });\n}\n\nfunction renderTable(companies, showScore) {\n  resultsBody.innerHTML = '';\n  companies.forEach((c, i) => {\n    const tr = document.createElement('tr');\n    const scoreCell = showScore\n      ? `<td><span class=\"score-pill\">${Math.round((c.score || 0) * 100)}%</span></td>`\n      : '';\n    tr.innerHTML = `\n      <td>${i + 1}</td>\n      <td class=\"company-name\">${esc(c.company_name)}</td>\n      <td>${esc(c.contact_person)}</td>\n      <td>${esc(c.phone1)}</td>\n      <td>${esc(c.phone2)}</td>\n      <td>${esc(c.mobile)}</td>\n      <td>${emailLink(c.email1)}</td>\n      <td>${emailLink(c.email2)}</td>\n      ${scoreCell}\n      <td><a class=\"detail-link\" href=\"/company/${c.id}\" target=\"_blank\">View \u2197</a></td>\n    `;\n    resultsBody.appendChild(tr);\n  });\n}\n\nfunction esc(val) {\n  if (!val) return '<span style=\"color:#94a3b8\">\u2014</span>';\n  return val.replace(/&/g,'&amp;').replace(/</g,'&lt;').replace(/>/g,'&gt;');\n}\nfunction emailLink(val) {\n  if (!val) return '<span style=\"color:#94a3b8\">\u2014</span>';\n  return `<a href=\"mailto:${val}\" style=\"color:#2563eb\">${val}</a>`;\n}\n\n// \u2500\u2500 Excel export \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\nif (exportBtn) {\n  exportBtn.addEventListener('click', () => {\n    if (!lastParams.country && currentMode === 'classic') return;\n    window.location.href = '/export?' + new URLSearchParams(lastParams);\n  });\n}\n\n// \u2500\u2500 Loader helpers \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\nfunction showLoader(text) {\n  if (loaderText) loaderText.textContent = text;\n  loader.style.display     = 'flex';\n  resultsSection.style.display = 'none';\n}\nfunction hideLoader() {\n  loader.style.display = 'none';\n}\n\n// \u2500\u2500 Chatbot \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\nlet chatHistory = [];\n\nfunction toggleChat(open) {\n  const panel = document.getElementById('chatPanel');\n  const fab   = document.getElementById('chatFab');\n  if (!panel) return;\n  if (open) {\n    panel.classList.add('open');\n    if (fab) fab.style.display = 'none';\n    setTimeout(() => document.getElementById('chatInput')?.focus(), 200);\n  } else {\n    panel.classList.remove('open');\n    if (fab) fab.style.display = 'flex';\n  }\n}\n\nasync function sendChat() {\n  const input   = document.getElementById('chatInput');\n  const messages = document.getElementById('chatMessages');\n  const text    = input.value.trim();\n  if (!text) return;\n\n  // Add user bubble\n  appendBubble(messages, text, 'user');\n  input.value = '';\n  chatHistory.push({ role: 'user', content: text });\n\n  // Typing indicator\n  const typingId = appendTyping(messages);\n\n  try {\n    const res  = await fetch('/ai/chat', {\n      method: 'POST',\n      headers: { 'Content-Type': 'application/json' },\n      body: JSON.stringify({ message: text, history: chatHistory })\n    });\n    const data = await res.json();\n    removeTyping(typingId, messages);\n\n    const reply = data.reply || 'Sorry, I could not process that.';\n    chatHistory.push({ role: 'assistant', content: reply });\n\n    const msgDiv = appendBubble(messages, reply, 'assistant');\n\n    // Show company links if any\n    if (data.companies && data.companies.length > 0) {\n      const linksDiv = document.createElement('div');\n      linksDiv.className = 'chat-companies';\n      data.companies.slice(0, 5).forEach(c => {\n        const a = document.createElement('a');\n        a.className = 'chat-company-link';\n        a.href      = `/company/${c.id}`;\n        a.target    = '_blank';\n        a.textContent = `${c.company_name} \u00b7 ${c.country}`;\n        linksDiv.appendChild(a);\n      });\n      if (data.companies.length > 5) {\n        const more = document.createElement('div');\n        more.style.cssText = 'font-size:12px;color:#64748b;padding:4px 0';\n        more.textContent   = `+ ${data.companies.length - 5} more companies`;\n        linksDiv.appendChild(more);\n      }\n      msgDiv.querySelector('.msg-bubble').appendChild(linksDiv);\n    }\n  } catch (err) {\n    removeTyping(typingId, messages);\n    appendBubble(messages, 'Connection error. Please try again.', 'assistant');\n  }\n}\n\nfunction appendBubble(container, text, role) {\n  const div = document.createElement('div');\n  div.className = `chat-msg ${role}`;\n  div.innerHTML = `<div class=\"msg-bubble\">${text.replace(/\\*\\*(.*?)\\*\\*/g,'<strong>$1</strong>').replace(/\\n/g,'<br>')}</div>`;\n  container.appendChild(div);\n  container.scrollTop = container.scrollHeight;\n  return div;\n}\n\nfunction appendTyping(container) {\n  const id  = 'typing-' + Date.now();\n  const div = document.createElement('div');\n  div.id        = id;\n  div.className = 'chat-msg assistant';\n  div.innerHTML = '<div class=\"msg-bubble\"><div class=\"typing-dots\"><span></span><span></span><span></span></div></div>';\n  container.appendChild(div);\n  container.scrollTop = container.scrollHeight;\n  return id;\n}\n\nfunction removeTyping(id, container) {\n  const el = document.getElementById(id);\n  if (el) el.remove();\n}\n</script>\n</body></html>")
pathlib.Path('/content/tradefind/templates/detail.html').write_text("<!DOCTYPE html>\n<html lang=\"en\">\n<head>\n<meta charset=\"UTF-8\">\n<title>{{ company.company_name }} - TradeFind</title>\n<link href=\"https://fonts.googleapis.com/css2?family=Syne:wght@700;800&family=DM+Sans:wght@300;400;500&display=swap\" rel=\"stylesheet\">\n<style>/* \u2500\u2500 Reset & Base \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500 */\n*, *::before, *::after { box-sizing: border-box; margin: 0; padding: 0; }\n:root {\n  --navy:    #0d1f3c;\n  --navy2:   #1b3a6b;\n  --blue:    #2563eb;\n  --blue-lt: #dbeafe;\n  --accent:  #f59e0b;\n  --green:   #10b981;\n  --red:     #ef4444;\n  --gray1:   #f8fafc;\n  --gray2:   #f1f5f9;\n  --gray3:   #e2e8f0;\n  --gray5:   #64748b;\n  --gray7:   #334155;\n  --gray9:   #0f172a;\n  --white:   #ffffff;\n  --font-head: 'Syne', sans-serif;\n  --font-body: 'DM Sans', sans-serif;\n  --radius:  12px;\n  --shadow:  0 4px 24px rgba(13,31,60,.10);\n  --shadow2: 0 8px 40px rgba(13,31,60,.16);\n}\n\nbody {\n  font-family: var(--font-body);\n  color: var(--gray9);\n  background: #f0f4fa;\n  min-height: 100vh;\n  line-height: 1.6;\n  position: relative;\n}\n\n/* \u2500\u2500 Background grid \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500 */\n.bg-grid {\n  position: fixed; inset: 0; z-index: 0; pointer-events: none;\n  background-image:\n    linear-gradient(rgba(37,99,235,.04) 1px, transparent 1px),\n    linear-gradient(90deg, rgba(37,99,235,.04) 1px, transparent 1px);\n  background-size: 48px 48px;\n}\n\n.container { max-width: 1200px; margin: 0 auto; padding: 0 24px; position: relative; z-index: 1; }\n\n/* \u2500\u2500 Header \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500 */\n.site-header {\n  background: var(--navy);\n  padding: 0 0;\n  position: sticky; top: 0; z-index: 100;\n  border-bottom: 1px solid rgba(255,255,255,.06);\n}\n.site-header .container { display: flex; align-items: center; justify-content: space-between; height: 60px; }\n\n.logo { display: flex; align-items: center; gap: 10px; text-decoration: none; color: var(--white); }\n.logo-mark { color: var(--accent); font-size: 18px; line-height: 1; }\n.logo-text  { font-family: var(--font-head); font-size: 20px; color: var(--white); }\n.logo-text strong { color: var(--accent); }\n\nnav a { color: rgba(255,255,255,.7); text-decoration: none; font-size: 14px; margin-left: 24px;\n        transition: color .2s; }\nnav a:hover { color: var(--white); }\n\n/* \u2500\u2500 Hero \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500 */\n.hero { padding: 64px 0 48px; }\n.hero-badge {\n  display: inline-block; background: var(--blue-lt); color: var(--blue);\n  font-size: 12px; font-weight: 500; letter-spacing: .08em; text-transform: uppercase;\n  padding: 4px 12px; border-radius: 20px; margin-bottom: 20px;\n}\n.hero h1 {\n  font-family: var(--font-head); font-size: clamp(36px, 5vw, 60px);\n  font-weight: 800; line-height: 1.12; color: var(--navy); margin-bottom: 16px;\n}\n.hero h1 em { font-style: normal; color: var(--blue); }\n.hero-sub { font-size: 18px; color: var(--gray5); max-width: 540px; margin-bottom: 40px; font-weight: 300; }\n\n/* \u2500\u2500 Search Card \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500 */\n.search-card {\n  background: var(--white);\n  border-radius: 20px;\n  box-shadow: var(--shadow2);\n  padding: 36px 40px;\n  border: 1px solid var(--gray3);\n}\n\n.form-grid { display: grid; grid-template-columns: 1fr 1fr 1fr; gap: 24px; margin-bottom: 28px; }\n@media (max-width: 768px) { .form-grid { grid-template-columns: 1fr; } }\n\n.form-group label {\n  display: block; font-size: 13px; font-weight: 500; color: var(--gray7);\n  margin-bottom: 8px; text-transform: uppercase; letter-spacing: .06em;\n}\n\n.select-wrap { position: relative; }\n.select-wrap::after {\n  content: '\u25be'; position: absolute; right: 14px; top: 50%;\n  transform: translateY(-50%); color: var(--gray5); pointer-events: none; font-size: 13px;\n}\nselect {\n  width: 100%; padding: 12px 40px 12px 16px;\n  border: 1.5px solid var(--gray3); border-radius: var(--radius);\n  font-family: var(--font-body); font-size: 15px; color: var(--gray9);\n  background: var(--gray1); appearance: none; cursor: pointer;\n  transition: border-color .2s, box-shadow .2s;\n}\nselect:focus { outline: none; border-color: var(--blue); box-shadow: 0 0 0 3px rgba(37,99,235,.12); }\n\n/* \u2500\u2500 Radio Toggle \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500 */\n.radio-group { display: flex; gap: 12px; }\n.radio-btn { flex: 1; }\n.radio-btn input { display: none; }\n.radio-btn span {\n  display: flex; align-items: center; justify-content: center;\n  padding: 12px; border: 1.5px solid var(--gray3); border-radius: var(--radius);\n  font-size: 15px; font-weight: 500; cursor: pointer; background: var(--gray1);\n  transition: all .2s;\n}\n.radio-btn input:checked + span {\n  background: var(--navy2); color: var(--white);\n  border-color: var(--navy2);\n}\n.radio-btn:hover span { border-color: var(--blue); }\n\n/* \u2500\u2500 Search Button \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500 */\n.btn-search {\n  display: inline-flex; align-items: center; gap: 10px;\n  background: var(--blue); color: var(--white);\n  border: none; border-radius: var(--radius);\n  font-family: var(--font-head); font-size: 16px; font-weight: 600;\n  padding: 14px 36px; cursor: pointer;\n  transition: background .2s, transform .15s, box-shadow .2s;\n  box-shadow: 0 4px 16px rgba(37,99,235,.35);\n}\n.btn-search:hover { background: #1d4ed8; transform: translateY(-1px); box-shadow: 0 6px 24px rgba(37,99,235,.45); }\n.btn-search:active { transform: translateY(0); }\n\n/* \u2500\u2500 Loader \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500 */\n.loader { text-align: center; padding: 60px 0; display: flex; flex-direction: column; align-items: center; gap: 16px; }\n.spinner {\n  width: 40px; height: 40px;\n  border: 3px solid var(--gray3);\n  border-top-color: var(--blue);\n  border-radius: 50%;\n  animation: spin .7s linear infinite;\n}\n@keyframes spin { to { transform: rotate(360deg); } }\n.loader p { color: var(--gray5); font-size: 14px; }\n\n/* \u2500\u2500 Results Section \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500 */\n.results-section { padding: 48px 0 80px; }\n.results-header { display: flex; align-items: center; justify-content: space-between; margin-bottom: 24px; flex-wrap: wrap; gap: 16px; }\n.results-header h2 { font-family: var(--font-head); font-size: 28px; font-weight: 700; color: var(--navy); }\n.results-meta { font-size: 14px; color: var(--gray5); margin-top: 2px; }\n\n.btn-export {\n  display: inline-flex; align-items: center; gap: 8px;\n  background: var(--green); color: var(--white);\n  border: none; border-radius: var(--radius);\n  font-family: var(--font-body); font-size: 14px; font-weight: 500;\n  padding: 10px 22px; cursor: pointer;\n  transition: background .2s, transform .15s;\n  text-decoration: none;\n}\n.btn-export:hover { background: #059669; transform: translateY(-1px); }\n\n/* \u2500\u2500 Table \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500 */\n.table-wrap { overflow-x: auto; border-radius: 16px; box-shadow: var(--shadow); }\n.results-table {\n  width: 100%; border-collapse: collapse;\n  background: var(--white); font-size: 14px;\n  border-radius: 16px; overflow: hidden;\n}\n.results-table thead tr { background: var(--navy); }\n.results-table th {\n  padding: 14px 16px; text-align: left;\n  color: rgba(255,255,255,.85); font-size: 12px;\n  font-weight: 600; letter-spacing: .06em; text-transform: uppercase;\n  white-space: nowrap;\n}\n.results-table tbody tr { border-bottom: 1px solid var(--gray3); transition: background .15s; }\n.results-table tbody tr:last-child { border-bottom: none; }\n.results-table tbody tr:hover { background: var(--blue-lt); }\n.results-table td { padding: 13px 16px; color: var(--gray7); vertical-align: middle; }\n.results-table td:first-child { font-weight: 600; color: var(--gray5); width: 48px; }\n.results-table .company-name { font-weight: 600; color: var(--navy); white-space: nowrap; }\na.detail-link {\n  display: inline-flex; align-items: center; gap: 4px;\n  color: var(--blue); font-weight: 500; text-decoration: none; font-size: 13px;\n  padding: 5px 12px; border: 1px solid var(--blue-lt); border-radius: 20px;\n  transition: background .15s;\n}\na.detail-link:hover { background: var(--blue-lt); }\n\n/* \u2500\u2500 No Results \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500 */\n.no-results {\n  background: var(--white); border-radius: 16px; text-align: center;\n  padding: 60px 40px; box-shadow: var(--shadow);\n}\n.no-results-icon { font-size: 36px; margin-bottom: 16px; }\n.no-results h3 { font-family: var(--font-head); font-size: 22px; color: var(--navy); margin-bottom: 8px; }\n.no-results p { color: var(--gray5); }\n\n/* \u2500\u2500 Detail Page \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500 */\n.detail-hero { padding: 56px 0 32px; }\n.detail-badges { display: flex; gap: 10px; margin-bottom: 16px; flex-wrap: wrap; }\n.badge {\n  display: inline-block; padding: 4px 14px; border-radius: 20px;\n  font-size: 12px; font-weight: 600; letter-spacing: .06em; text-transform: uppercase;\n}\n.badge-type.export { background: #dcfce7; color: #166534; }\n.badge-type.import { background: #dbeafe; color: #1e40af; }\n.badge-product  { background: #fef3c7; color: #92400e; }\n.badge-country  { background: var(--gray2); color: var(--gray7); }\n\n.detail-title { font-family: var(--font-head); font-size: clamp(28px,4vw,46px); font-weight: 800; color: var(--navy); margin-bottom: 14px; }\n\n.website-link {\n  display: inline-flex; align-items: center; gap: 6px;\n  color: var(--blue); text-decoration: none; font-size: 14px; font-weight: 500;\n  padding: 6px 16px; border: 1.5px solid var(--blue-lt); border-radius: 20px;\n  transition: background .15s;\n}\n.website-link:hover { background: var(--blue-lt); }\n\n.detail-body { padding: 0 0 80px; }\n.detail-grid { display: grid; grid-template-columns: 1fr 1fr; gap: 20px; margin-bottom: 32px; }\n@media (max-width: 768px) { .detail-grid { grid-template-columns: 1fr; } }\n.full-width { grid-column: 1 / -1; }\n\n.detail-card {\n  background: var(--white); border-radius: 16px; border: 1px solid var(--gray3);\n  box-shadow: var(--shadow); overflow: hidden;\n}\n.card-header {\n  display: flex; align-items: center; gap: 12px;\n  padding: 18px 24px; border-bottom: 1px solid var(--gray3);\n  background: var(--gray1);\n}\n.card-icon { font-size: 18px; }\n.card-header h3 { font-family: var(--font-head); font-size: 16px; font-weight: 700; color: var(--navy); }\n.card-body { padding: 20px 24px; }\n\n.info-row { display: flex; align-items: baseline; justify-content: space-between; gap: 16px; padding: 10px 0; border-bottom: 1px solid var(--gray2); }\n.info-row:last-child { border-bottom: none; }\n.info-label { font-size: 13px; color: var(--gray5); font-weight: 500; white-space: nowrap; }\n.info-value { font-size: 14px; color: var(--gray9); font-weight: 500; text-align: right; }\n.info-value a { color: var(--blue); text-decoration: none; }\n.info-value a:hover { text-decoration: underline; }\n\n.address-text { font-size: 15px; color: var(--gray7); line-height: 1.7; margin-bottom: 16px; }\n.map-link { color: var(--blue); font-size: 13px; font-weight: 500; text-decoration: none; }\n.map-link:hover { text-decoration: underline; }\n\n.two-col { display: grid; grid-template-columns: 1fr 1fr; gap: 24px; }\n@media (max-width: 640px) { .two-col { grid-template-columns: 1fr; } }\n.two-col h4 { font-family: var(--font-head); font-size: 14px; font-weight: 700; color: var(--navy); margin-bottom: 12px; text-transform: uppercase; letter-spacing: .06em; }\n\n.tag-list { display: flex; flex-wrap: wrap; gap: 8px; }\n.tag {\n  background: var(--blue-lt); color: #1e40af;\n  padding: 5px 14px; border-radius: 20px; font-size: 13px; font-weight: 500;\n}\n.tag-service { background: #f0fdf4; color: #166534; }\n\n.detail-actions { display: flex; gap: 16px; flex-wrap: wrap; }\n.btn-back {\n  display: inline-flex; align-items: center; gap: 6px;\n  padding: 12px 24px; border-radius: var(--radius);\n  border: 1.5px solid var(--gray3); color: var(--gray7);\n  font-family: var(--font-body); font-size: 14px; font-weight: 500;\n  text-decoration: none; transition: border-color .2s, background .2s;\n}\n.btn-back:hover { border-color: var(--blue); color: var(--blue); background: var(--blue-lt); }\n.btn-visit {\n  display: inline-flex; align-items: center; gap: 6px;\n  padding: 12px 24px; border-radius: var(--radius);\n  background: var(--blue); color: var(--white);\n  font-family: var(--font-body); font-size: 14px; font-weight: 500;\n  text-decoration: none; transition: background .2s, transform .15s;\n}\n.btn-visit:hover { background: #1d4ed8; transform: translateY(-1px); }\n\n/* \u2500\u2500 AI Badge \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500 */\n.ai-badge {\n  display: inline-flex; align-items: center; gap: 8px;\n  background: linear-gradient(135deg, #1e1b4b, #1e3a5f);\n  color: #a5b4fc; font-size: 13px; font-weight: 500;\n  padding: 8px 16px; border-radius: 20px; margin-bottom: 32px;\n  border: 1px solid rgba(165,180,252,.25);\n}\n.ai-dot {\n  width: 8px; height: 8px; border-radius: 50%;\n  background: #818cf8;\n  box-shadow: 0 0 0 3px rgba(129,140,248,.25);\n  animation: pulse 2s ease-in-out infinite;\n  flex-shrink: 0;\n}\n@keyframes pulse { 0%,100%{box-shadow:0 0 0 3px rgba(129,140,248,.25)} 50%{box-shadow:0 0 0 6px rgba(129,140,248,.10)} }\n\n/* \u2500\u2500 Score bar \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500 */\n.score-bar-wrap {\n  display: flex; align-items: center; gap: 6px;\n  width: 90px; cursor: help;\n}\n.score-bar {\n  height: 6px; border-radius: 3px;\n  transition: width .4s ease;\n  flex-shrink: 0;\n}\n.score-label { font-size: 12px; font-weight: 600; white-space: nowrap; }\n\n/* \u2500\u2500 Footer \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500 */\n.site-footer {\n  background: var(--navy); color: rgba(255,255,255,.45);\n  padding: 24px 0; text-align: center; font-size: 13px;\n}\n\n/* \u2500\u2500 AI Badge \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500 */\n.ai-badge {\n  background: linear-gradient(90deg,#7f77dd,#2563eb);\n  color:#fff; font-size:11px; font-weight:600; letter-spacing:.06em;\n  padding:3px 10px; border-radius:20px; margin-left:10px; vertical-align:middle;\n}\n.ai-label {\n  background:var(--blue-lt); color:var(--blue); font-size:11px; font-weight:600;\n  padding:2px 10px; border-radius:20px; margin-left:8px; letter-spacing:.04em;\n}\n\n/* \u2500\u2500 Search Mode Toggle \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500 */\n.search-mode-toggle { display:flex; gap:8px; margin-bottom:16px; }\n.mode-btn {\n  padding:8px 20px; border-radius:20px; border:1.5px solid var(--gray3);\n  font-family:var(--font-body); font-size:14px; font-weight:500; cursor:pointer;\n  background:var(--white); color:var(--gray5); transition:all .2s;\n}\n.mode-btn.active { background:var(--navy); color:var(--white); border-color:var(--navy); }\n.mode-btn:hover:not(.active) { border-color:var(--blue); color:var(--blue); }\n\n/* \u2500\u2500 AI Search Card \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500 */\n.ai-card { border:1.5px solid #c4b5fd; background:#faf9ff; }\n.ai-card-header { display:flex; align-items:flex-start; gap:14px; margin-bottom:20px; }\n.ai-icon-lg { font-size:22px; color:#7f77dd; line-height:1; margin-top:2px; }\n.ai-card-title { font-family:var(--font-head); font-size:17px; font-weight:700; color:var(--navy); }\n.ai-card-sub { font-size:13px; color:var(--gray5); margin-top:3px; }\n.ai-form-row { display:flex; gap:12px; flex-wrap:wrap; align-items:stretch; }\n.ai-input {\n  flex:1; min-width:220px; padding:12px 16px;\n  border:1.5px solid var(--gray3); border-radius:var(--radius);\n  font-family:var(--font-body); font-size:15px; color:var(--gray9); background:var(--white);\n  transition:border-color .2s, box-shadow .2s;\n}\n.ai-input:focus { outline:none; border-color:#7f77dd; box-shadow:0 0 0 3px rgba(127,119,221,.12); }\n.ai-select-wrap { min-width:140px; }\n\n/* \u2500\u2500 Score column \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500 */\n.score-pill {\n  display:inline-block; padding:3px 10px; border-radius:20px;\n  font-size:12px; font-weight:600; background:#ede9fe; color:#4c1d95;\n}\n\n/* \u2500\u2500 Chatbot FAB \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500 */\n.chat-fab {\n  position:fixed; bottom:28px; right:28px; z-index:200;\n  display:flex; align-items:center; gap:8px;\n  background:var(--navy); color:var(--white);\n  border:none; border-radius:30px; padding:12px 22px;\n  font-family:var(--font-body); font-size:14px; font-weight:500; cursor:pointer;\n  box-shadow:0 4px 20px rgba(13,31,60,.3); transition:transform .2s, box-shadow .2s;\n}\n.chat-fab:hover { transform:translateY(-2px); box-shadow:0 6px 28px rgba(13,31,60,.4); }\n\n/* \u2500\u2500 Chatbot Panel \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500 */\n.chat-panel {\n  position:fixed; bottom:90px; right:28px; z-index:300;\n  width:380px; max-height:520px;\n  background:var(--white); border-radius:20px;\n  box-shadow:0 12px 48px rgba(13,31,60,.2);\n  border:1px solid var(--gray3);\n  display:none; flex-direction:column; overflow:hidden;\n}\n.chat-panel.open { display:flex; }\n.chat-header {\n  display:flex; align-items:center; justify-content:space-between;\n  padding:16px 20px; background:var(--navy); color:var(--white);\n}\n.chat-title { display:flex; align-items:center; gap:8px; font-family:var(--font-head); font-size:15px; font-weight:700; }\n.ai-icon { color:#a78bfa; font-size:16px; }\n.chat-close { background:none; border:none; color:rgba(255,255,255,.6); font-size:18px; cursor:pointer; line-height:1; }\n.chat-close:hover { color:var(--white); }\n.chat-messages { flex:1; overflow-y:auto; padding:16px; display:flex; flex-direction:column; gap:12px; }\n.chat-msg { display:flex; }\n.chat-msg.assistant .msg-bubble {\n  background:var(--gray1); border:1px solid var(--gray3); border-radius:4px 16px 16px 16px;\n  padding:12px 14px; font-size:14px; color:var(--gray7); line-height:1.6; max-width:90%;\n}\n.chat-msg.user { justify-content:flex-end; }\n.chat-msg.user .msg-bubble {\n  background:var(--navy); color:var(--white);\n  border-radius:16px 4px 16px 16px;\n  padding:10px 14px; font-size:14px; max-width:85%;\n}\n.chat-companies { margin-top:10px; display:flex; flex-direction:column; gap:6px; }\n.chat-company-link {\n  display:block; padding:8px 12px; border-radius:8px;\n  background:var(--white); border:1px solid var(--gray3);\n  font-size:13px; color:var(--navy); text-decoration:none; font-weight:500;\n  transition:background .15s;\n}\n.chat-company-link:hover { background:var(--blue-lt); }\n.chat-input-row {\n  display:flex; gap:8px; padding:12px 16px;\n  border-top:1px solid var(--gray3);\n}\n.chat-input-row input {\n  flex:1; padding:10px 14px; border:1.5px solid var(--gray3);\n  border-radius:20px; font-family:var(--font-body); font-size:14px;\n  transition:border-color .2s;\n}\n.chat-input-row input:focus { outline:none; border-color:var(--blue); }\n.btn-send {\n  width:40px; height:40px; border-radius:50%; background:var(--blue); color:var(--white);\n  border:none; cursor:pointer; display:flex; align-items:center; justify-content:center;\n  transition:background .2s;\n}\n.btn-send:hover { background:#1d4ed8; }\n.typing-dots span {\n  display:inline-block; width:6px; height:6px; border-radius:50%;\n  background:var(--gray5); margin:0 2px;\n  animation:bounce .9s infinite; \n}\n.typing-dots span:nth-child(2) { animation-delay:.15s; }\n.typing-dots span:nth-child(3) { animation-delay:.3s; }\n@keyframes bounce { 0%,80%,100%{transform:translateY(0)} 40%{transform:translateY(-6px)} }\n\n/* \u2500\u2500 Similar Companies \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500 */\n.similar-grid { display:grid; grid-template-columns:1fr 1fr; gap:12px; }\n@media(max-width:640px){ .similar-grid{grid-template-columns:1fr;} }\n.similar-card {\n  display:block; padding:14px 16px; border-radius:12px;\n  border:1.5px solid var(--gray3); text-decoration:none;\n  transition:border-color .2s, background .2s;\n}\n.similar-card:hover { border-color:var(--blue); background:var(--blue-lt); }\n.similar-name { font-weight:600; color:var(--navy); font-size:14px; margin-bottom:4px; }\n.similar-meta { font-size:12px; color:var(--gray5); margin-bottom:10px; }\n.similar-score { display:flex; align-items:center; gap:8px; }\n.score-bar { flex:1; height:5px; background:var(--gray3); border-radius:3px; overflow:hidden; }\n.score-fill { height:100%; background:var(--blue); border-radius:3px; }\n.similar-score span { font-size:11px; color:var(--gray5); font-weight:500; white-space:nowrap; }\n</style>\n</head>\n<body>\n<div class=\"bg-grid\"></div>\n<header class=\"site-header\"><div class=\"container\"><div class=\"logo\"><span class=\"logo-mark\">TF</span><span class=\"logo-text\">Trade<strong>Find</strong></span></div><nav><a href=\"/\">Back to Search</a></nav></div></header>\n<main>\n  <section class=\"detail-hero\"><div class=\"container\">\n    <div class=\"detail-badges\"><span class=\"badge badge-type {{ company.trade_type }}\">{{ company.trade_type|upper }}ER</span><span class=\"badge badge-product\">{{ company.product }}</span><span class=\"badge badge-country\">{{ company.country }}</span></div>\n    <h1 class=\"detail-title\">{{ company.company_name }}</h1>\n    {% if company.website %}<a href=\"{{ company.website }}\" target=\"_blank\" class=\"website-link\">{{ company.website }}</a>{% endif %}\n  </div></section>\n  <section class=\"detail-body\"><div class=\"container\"><div class=\"detail-grid\">\n    <div class=\"detail-card\"><div class=\"card-header\"><span class=\"card-icon\">Contact</span><h3>Contact Information</h3></div><div class=\"card-body\">\n      <div class=\"info-row\"><span class=\"info-label\">Contact Person</span><span class=\"info-value\">{{ company.contact_person or '-' }}</span></div>\n      <div class=\"info-row\"><span class=\"info-label\">Phone 1</span><span class=\"info-value\">{{ company.phone1 or '-' }}</span></div>\n      <div class=\"info-row\"><span class=\"info-label\">Phone 2</span><span class=\"info-value\">{{ company.phone2 or '-' }}</span></div>\n      <div class=\"info-row\"><span class=\"info-label\">Mobile</span><span class=\"info-value\">{{ company.mobile or '-' }}</span></div>\n      <div class=\"info-row\"><span class=\"info-label\">Email 1</span><span class=\"info-value\">{% if company.email1 %}<a href=\"mailto:{{ company.email1 }}\">{{ company.email1 }}</a>{% else %}-{% endif %}</span></div>\n      <div class=\"info-row\"><span class=\"info-label\">Email 2</span><span class=\"info-value\">{% if company.email2 %}<a href=\"mailto:{{ company.email2 }}\">{{ company.email2 }}</a>{% else %}-{% endif %}</span></div>\n    </div></div>\n    <div class=\"detail-card\"><div class=\"card-header\"><span class=\"card-icon\">Addr</span><h3>Address</h3></div><div class=\"card-body\">\n      <p class=\"address-text\">{{ company.address or 'Not available' }}</p>\n      {% if company.address %}<a href=\"https://maps.google.com/?q={{ company.address|urlencode }}\" target=\"_blank\" class=\"map-link\">Open in Google Maps</a>{% endif %}\n    </div></div>\n    <div class=\"detail-card full-width\"><div class=\"card-header\"><span class=\"card-icon\">Prod</span><h3>Products and Services</h3></div><div class=\"card-body two-col\">\n      <div><h4>Products</h4>{% if company.products %}<div class=\"tag-list\">{% for p in company.products.split(',') %}<span class=\"tag\">{{ p.strip() }}</span>{% endfor %}</div>{% else %}<p>-</p>{% endif %}</div>\n      <div><h4>Services</h4>{% if company.services %}<div class=\"tag-list\">{% for s in company.services.split(',') %}<span class=\"tag tag-service\">{{ s.strip() }}</span>{% endfor %}</div>{% else %}<p>-</p>{% endif %}</div>\n    </div></div>\n    {% if similar %}\n    <div class=\"detail-card full-width\"><div class=\"card-header\"><h3>Similar Companies <span class=\"ai-label\">AI Recommended</span></h3></div><div class=\"card-body\">\n      <div class=\"similar-grid\">{% for s in similar %}<a href=\"/company/{{ s.id }}\" class=\"similar-card\"><div class=\"similar-name\">{{ s.company_name }}</div><div class=\"similar-meta\">{{ s.product }} - {{ s.country }}</div><div class=\"similar-score\"><div class=\"score-bar\"><div class=\"score-fill\" style=\"width:{{ (s.similarity*100)|int }}%\"></div></div><span>{{ (s.similarity*100)|int }}% match</span></div></a>{% endfor %}</div>\n    </div></div>{% endif %}\n  </div>\n  <div class=\"detail-actions\"><a href=\"/\" class=\"btn-back\">New Search</a>{% if company.website %}<a href=\"{{ company.website }}\" target=\"_blank\" class=\"btn-visit\">Visit Website</a>{% endif %}</div>\n  </div></section>\n</main>\n<footer class=\"site-footer\"><div class=\"container\"><p>2025 TradeFind</p></div></footer>\n</body></html>")
print('4/6 Templates ready')

# 5 — Flask app
pathlib.Path('/content/tradefind/app.py').write_text("import sys\nsys.path.insert(0, \"/content/tradefind\")\nfrom flask import Flask, render_template, request, jsonify, send_file\nimport sqlite3, openpyxl, io, os\nfrom openpyxl.styles import Font, PatternFill, Alignment, Border, Side\n\napp = Flask(__name__, template_folder=\"/content/tradefind/templates\", static_folder=\"/content/tradefind/static\")\nDB = \"/content/tradefind/data/trade.db\"\n\ndef _load_ai():\n    try:\n        from ai.nlp_search  import build_index as bn, semantic_search\n        from ai.recommender import build_index as br, get_similar\n        bn(); br(); print(\"[App] AI ready!\"); return semantic_search, get_similar\n    except Exception as e: print(f\"[App] AI skipped: {e}\"); return None, None\n\nss, gs = _load_ai()\n\ndef db():\n    c = sqlite3.connect(DB); c.row_factory = sqlite3.Row; return c\n\n@app.route(\"/\")\ndef index():\n    c = db()\n    countries = [r[\"country\"] for r in c.execute(\"SELECT DISTINCT country FROM companies ORDER BY country\").fetchall()]\n    products  = [r[\"product\"]  for r in c.execute(\"SELECT DISTINCT product  FROM companies ORDER BY product\").fetchall()]\n    c.close()\n    return render_template(\"index.html\", countries=countries, products=products, ai_enabled=ss is not None)\n\n@app.route(\"/search\")\ndef search():\n    co=request.args.get(\"country\",\"\").strip(); pr=request.args.get(\"product\",\"\").strip(); tt=request.args.get(\"trade_type\",\"\").strip()\n    c=db(); q=\"SELECT * FROM companies WHERE 1=1\"; p=[]\n    if co: q+=\" AND LOWER(country)=LOWER(?)\"; p.append(co)\n    if pr: q+=\" AND LOWER(product)=LOWER(?)\"; p.append(pr)\n    if tt: q+=\" AND LOWER(trade_type)=LOWER(?)\"; p.append(tt)\n    rows=c.execute(q,p).fetchall(); c.close()\n    return jsonify([dict(r) for r in rows])\n\n@app.route(\"/company/<int:cid>\")\ndef detail(cid):\n    c=db(); row=c.execute(\"SELECT * FROM companies WHERE id=?\",(cid,)).fetchone(); c.close()\n    if not row: return \"Not found\", 404\n    sim=[]\n    if gs:\n        try: sim=gs(cid, top_k=4)\n        except: pass\n    return render_template(\"detail.html\", company=dict(row), similar=sim)\n\n@app.route(\"/export\")\ndef export():\n    co=request.args.get(\"country\",\"\"); pr=request.args.get(\"product\",\"\"); tt=request.args.get(\"trade_type\",\"\")\n    c=db(); q=\"SELECT * FROM companies WHERE 1=1\"; p=[]\n    if co: q+=\" AND LOWER(country)=LOWER(?)\"; p.append(co)\n    if pr: q+=\" AND LOWER(product)=LOWER(?)\"; p.append(pr)\n    if tt: q+=\" AND LOWER(trade_type)=LOWER(?)\"; p.append(tt)\n    rows=c.execute(q,p).fetchall(); c.close()\n    wb=openpyxl.Workbook(); ws=wb.active; ws.title=\"Trade Directory\"\n    hdrs=[\"SR#\",\"Company Name\",\"Contact Person\",\"Phone 1\",\"Phone 2\",\"Mobile\",\"Email 1\",\"Email 2\"]\n    hf=PatternFill(\"solid\",fgColor=\"1B3A6B\"); hn=Font(bold=True,color=\"FFFFFF\",size=11)\n    af=PatternFill(\"solid\",fgColor=\"EEF3FB\"); bs=Side(style=\"thin\",color=\"C5CDD9\")\n    cb=Border(left=bs,right=bs,top=bs,bottom=bs); cw=[6,32,24,18,18,18,28,28]\n    for i,(h,w) in enumerate(zip(hdrs,cw),1):\n        cell=ws.cell(row=1,column=i,value=h); cell.font=hn; cell.fill=hf\n        cell.alignment=Alignment(horizontal=\"center\",vertical=\"center\"); cell.border=cb\n        ws.column_dimensions[cell.column_letter].width=w\n    ws.row_dimensions[1].height=22\n    for sr,row in enumerate(rows,1):\n        d=[sr,row[\"company_name\"],row[\"contact_person\"],row[\"phone1\"],row[\"phone2\"],row[\"mobile\"],row[\"email1\"],row[\"email2\"]]\n        for col,val in enumerate(d,1):\n            cell=ws.cell(row=sr+1,column=col,value=val or \"\"); cell.alignment=Alignment(vertical=\"center\"); cell.border=cb\n            if sr%2==0: cell.fill=af\n    ws.freeze_panes=\"A2\"; ws.auto_filter.ref=ws.dimensions\n    buf=io.BytesIO(); wb.save(buf); buf.seek(0)\n    return send_file(buf, as_attachment=True, download_name=f\"trade_{tt}_{pr}_{co}.xlsx\".replace(\" \",\"_\"),\n                     mimetype=\"application/vnd.openxmlformats-officedocument.spreadsheetml.sheet\")\n\n@app.route(\"/ai/search\")\ndef ai_search():\n    if not ss: return jsonify({\"error\":\"AI not available\"}), 503\n    q=request.args.get(\"q\",\"\").strip()\n    if not q: return jsonify({\"error\":\"q required\"}), 400\n    try: return jsonify(ss(q, country=request.args.get(\"country\",\"\"), trade_type=request.args.get(\"trade_type\",\"\")))\n    except Exception as e: return jsonify({\"error\":str(e)}), 500\n\n@app.route(\"/ai/similar/<int:cid>\")\ndef ai_similar(cid):\n    if not gs: return jsonify([])\n    try: return jsonify(gs(cid, top_k=4))\n    except Exception as e: return jsonify({\"error\":str(e)}), 500\n\n@app.route(\"/ai/chat\", methods=[\"POST\"])\ndef ai_chat():\n    from ai.chatbot import chat\n    d=request.get_json(silent=True) or {}; msg=d.get(\"message\",\"\").strip()\n    if not msg: return jsonify({\"error\":\"message required\"}), 400\n    return jsonify(chat(msg, history=d.get(\"history\",[])))\n\nif __name__ == \"__main__\":\n    app.run(port=5000)\n")
print('5/6 Flask app ready')

# 6 — Launch Flask
subprocess.run(['pkill','-f','app.py'], capture_output=True); time.sleep(1)
def run_flask(): subprocess.run([sys.executable,'/content/tradefind/app.py'])
threading.Thread(target=run_flask, daemon=True).start()
time.sleep(4)
print('6/6 Flask running!')
print()
print('Now run Cell 3 to get your public URL!')


In [ ]:
# ═══════════════════════════════════════════════════
# CELL 3 — Get your live public URL
# Wait ~15 seconds for the URL to appear
# ═══════════════════════════════════════════════════
import subprocess, threading, time, re

subprocess.run(['pkill','-f','cloudflared'], capture_output=True)
time.sleep(1)

with open('/tmp/tunnel.log','w') as log:
    subprocess.Popen(
        ['/usr/local/bin/cloudflared','tunnel','--url','http://localhost:5000'],
        stdout=log, stderr=log
    )

print('Starting tunnel, waiting 15 seconds...')
time.sleep(15)

with open('/tmp/tunnel.log','r') as f:
    content = f.read()

urls = re.findall(r'https://[a-z0-9\-]+\.trycloudflare\.com', content)

if urls:
    print()
    print('COPY THIS URL AND OPEN IN YOUR BROWSER:')
    print('=' * 55)
    print(urls[0])
    print('=' * 55)
    print('Classic : UAE + Oil + Exporter')
    print('AI      : type petroleum Gulf')
    print('Chatbot : oil exporters UAE with email')
else:
    print('URL not found yet. Wait 10 more seconds then run this cell again.')
    print('Last log output:')
    print(content[-1000:])


In [ ]:
# ═══════════════════════════════════════════════════
# CELL 4 — Stop the app (run when done)
# ═══════════════════════════════════════════════════
import subprocess
subprocess.run(['pkill','-f','cloudflared'], capture_output=True)
subprocess.run(['pkill','-f','app.py'], capture_output=True)
print('App stopped. Run Cell 2 then Cell 3 to restart.')
